# DiscoveryStack SEO/GEO 101：Google Colab 本地多任務訓練

本 notebook 僅接受 owner-approved immutable manifest 匯出的真實 JSONL。它不會建立假資料、放寬 PII gate 或重新切分資料。

In [ ]:
!pip -q install transformers datasets accelerate scikit-learn sentencepiece
!nvidia-smi || true

In [ ]:
from google.colab import files
uploaded = files.upload()
DATA_PATH = next(iter(uploaded))
print('uploaded:', DATA_PATH)

In [ ]:
import json, hashlib, os, random, shutil, zipfile
from collections import Counter
from pathlib import Path

REQUIRED_STAGES = {'discovery','understanding','response','progression','conversion'}
SPLITS = {'train','validation','test'}
TASKS = ['journeyStage','searchIntents','contentTypes','audienceRoles','geoSignals','citationReadiness','technicalSeoSignals','frictionSignals','actionPriority']

def read_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

rows = read_jsonl(DATA_PATH)
if len(rows) != 101:
    raise ValueError(f'FAIL-CLOSED: expected exactly 101 manifest members, found {len(rows)}')
ids = [int(r['id']) for r in rows]
if len(set(ids)) != len(ids):
    raise ValueError('FAIL-CLOSED: duplicate artifact IDs')
for i, r in enumerate(rows):
    text = r.get('trainingText') or r.get('text')
    if not text or not isinstance(text, str): raise ValueError(f'missing training text at row {i}')
    if r.get('split') not in SPLITS: raise ValueError(f'missing manifest split at row {i}')
    targets = r.get('targets') or {}
    missing = [t for t in TASKS if t not in targets]
    if missing: raise ValueError(f'missing multi-task targets {missing} at row {i}')
    if r.get('manifestHash') is None: raise ValueError(f'missing manifestHash at row {i}')
stages = Counter((r.get('targets') or {}).get('journeyStage') for r in rows)
if set(stages) != REQUIRED_STAGES or min(stages.values()) < 10:
    raise ValueError(f'FAIL-CLOSED: journey-stage distribution is {dict(stages)}')
manifest_hashes = {str(r['manifestHash']) for r in rows}
if len(manifest_hashes) != 1: raise ValueError(f'FAIL-CLOSED: mixed manifest hashes {manifest_hashes}')
MANIFEST_HASH = next(iter(manifest_hashes))
print('validated 101 rows; manifest hash:', MANIFEST_HASH)
print('stages:', dict(stages), 'splits:', dict(Counter(r['split'] for r in rows)))

In [ ]:
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, Trainer, TrainingArguments
import torch.nn as nn

MODEL_ID = 'distilbert-base-multilingual-cased'
SEED = 20260820
random.seed(SEED); torch.manual_seed(SEED)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def values(targets, task):
    v = targets[task]
    return v if isinstance(v, list) else [v]

label_maps = {}
for task in TASKS:
    vocab = sorted({str(v) for r in rows for v in values(r['targets'], task)})
    label_maps[task] = {v:i for i,v in enumerate(vocab)}

def encode(row):
    targets = row['targets']
    encoded = {'id': int(row['id']), 'split': row['split'], 'text': row.get('trainingText') or row.get('text')}
    for task in TASKS:
        vals = values(targets, task)
        encoded[task] = [label_maps[task][str(v)] for v in vals]
    return encoded
encoded = [encode(r) for r in rows]
train_rows = [r for r in encoded if r['split']=='train']
val_rows = [r for r in encoded if r['split']=='validation']
test_rows = [r for r in encoded if r['split']=='test']
print({k:len(v) for k,v in {'train':train_rows,'validation':val_rows,'test':test_rows}.items()})

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(self, model_id, label_maps):
        super().__init__(); self.encoder = AutoModel.from_pretrained(model_id)
        h = self.encoder.config.hidden_size
        self.heads = nn.ModuleDict({t: nn.Linear(h, len(label_maps[t])) for t in TASKS})
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state[:,0]
        logits = {t:self.heads[t](out) for t in TASKS}; loss=None
        if labels is not None:
            losses=[]
            for t in TASKS:
                y=labels[t]
                if y.ndim==1: y=y.unsqueeze(1)
                multi=torch.zeros((y.size(0), self.heads[t].out_features), device=y.device)
                multi.scatter_(1, y.clamp_min(0), 1.0)
                losses.append(nn.functional.binary_cross_entropy_with_logits(logits[t], multi))
            loss=torch.stack(losses).mean()
        return {'loss':loss,'logits':logits} if loss is not None else {'logits':logits}

def collate(batch):
    texts=[x['text'] for x in batch]
    tok=tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors='pt')
    for t in TASKS:
        tok['labels_'+t]=torch.tensor([x[t][0] for x in batch], dtype=torch.long)
    tok['labels']={t:tok.pop('labels_'+t) for t in TASKS}
    return tok

model=MultiTaskModel(MODEL_ID, label_maps)
args=TrainingArguments(output_dir='checkpoint', seed=SEED, num_train_epochs=3, per_device_train_batch_size=8, per_device_eval_batch_size=8, learning_rate=2e-5, weight_decay=0.01, evaluation_strategy='epoch', save_strategy='epoch', logging_steps=5, report_to='none', fp16=torch.cuda.is_available())
trainer=Trainer(model=model, args=args, train_dataset=train_rows, eval_dataset=val_rows, data_collator=collate)
trainer.train()

In [ ]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def evaluate(split_rows):
    pred=trainer.predict(split_rows)
    metrics={}
    for j,t in enumerate(TASKS):
        logits=pred.predictions['logits'][t] if isinstance(pred.predictions, dict) else pred.predictions[j]
        y=np.array([r[t][0] for r in split_rows]); p=logits.argmax(-1)
        metrics[t+'_accuracy']=float(accuracy_score(y,p)); metrics[t+'_macro_f1']=float(f1_score(y,p,average='macro',zero_division=0))
    return metrics
metrics={'validation':evaluate(val_rows),'test':evaluate(test_rows)}
out=Path('colab-training-artifacts'); out.mkdir(exist_ok=True)
config={'provider':'google_colab_local','model':MODEL_ID,'seed':SEED,'taskHeads':TASKS,'manifestHash':MANIFEST_HASH,'exampleCount':len(rows),'splitCounts':dict(Counter(r['split'] for r in rows)),'labelMaps':label_maps}
json.dump(config,open(out/'training-config.json','w'),ensure_ascii=False,indent=2); json.dump(metrics,open(out/'metrics.json','w'),ensure_ascii=False,indent=2)
trainer.save_model(out/'checkpoint'); tokenizer.save_pretrained(out/'checkpoint')
with zipfile.ZipFile('colab-training-artifacts.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in out.rglob('*'):
        if p.is_file(): z.write(p, p.as_posix())
print(json.dumps(metrics,ensure_ascii=False,indent=2)); files.download('colab-training-artifacts.zip')